# BTC Spot Grid Trading Backtest

V0 → V1 Dynamic + Risk → System Audit → GitHub Log. Manual checks: `Manual_logic_checker.ipynb`; automated tests: `tests/test_grid_trading.py`. Live orders OFF.

## 1. Setup & Engines

In [ ]:
import os,heapq,bisect,json,base64
import numpy as np,pandas as pd
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR='/content/drive/MyDrive/03.Trading/00.Live Trading';SYMBOL='BTCUSDT';START_DATE='2024-01-01';END_DATE='2026-01-01';BUY_FEE=SELL_FEE=.001
def load_market_data(s,t,p):
 p=os.path.join(p,f'{s}-{t}-combined.csv');d=pd.read_csv(p);d.open_time=pd.to_datetime(d.open_time,utc=True);d[['open','high','low','close','volume']]=d[['open','high','low','close','volume']].astype(float);return d.drop_duplicates('open_time').sort_values('open_time').reset_index(drop=True)
def build_excel_grid_table(capital,ceiling,floor,gap,buy_fee=.001,sell_fee=.001):
 n0=(ceiling-floor)/gap
 if capital<=0 or gap<=0 or ceiling<=floor or not np.isclose(n0,round(n0)):raise ValueError('Invalid grid inputs.')
 n=int(round(n0));c=capital/n;b=ceiling-gap*np.arange(1,n+1);s=b+gap;g=c/b;bf=g*buy_fee;ba=g-bf;gs=ba*s;sf=gs*sell_fee;net=gs-sf
 return pd.DataFrame({'level':np.arange(1,n+1),'buy_price':b,'sell_price':s,'capital_per_level':c,'gross_base_amount':g,'buy_fee_base':bf,'base_amount':ba,'gross_sell':gs,'sell_fee_quote':sf,'net_sell':net,'profit':net-c})
def _stats(d,e,p):
 pk=np.maximum.accumulate(e);dd=e/pk-1;f=float(e[-1]);days=(d.open_time.iloc[-1]-d.open_time.iloc[0]).total_seconds()/86400;lg=np.log(f/p)*(365.25/days) if days>0 and f>0 else np.nan;a=float(np.expm1(lg)) if np.isfinite(lg) and lg<700 else np.nan;m=float(dd.min());return f,f/p-1,a,m,a/abs(m) if m<0 and np.isfinite(a) else np.nan,dd
def run_grid_backtest(df_price,grid_table,initial_capital):
 d=df_price.sort_values('open_time').reset_index(drop=True);g=grid_table.sort_values('buy_price').reset_index(drop=True).copy()
 b,s,c,ba,bf,sf,net,p,l=[g[x].to_numpy(float if x!='level' else int) for x in ['buy_price','sell_price','capital_per_level','base_amount','buy_fee_base','sell_fee_quote','net_sell','profit','level']]
 h=np.zeros(len(g),bool);bt=[None]*len(g);hp=[];cash=float(initial_capital);btc=real=bfb=bfq=sfq=0.;cy=eid=0;ev=[];done=[];prev=None;eq=np.empty(len(d));cv=np.empty(len(d));bv=np.empty(len(d));bl=b.tolist()
 for i,r in enumerate(d.itertuples(index=False)):
  t,o,hi,lo,cl=r.open_time,float(r.open),float(r.high),float(r.low),float(r.close);start=cash;sold=set()
  while hp and hp[0][0]<=hi:
   _,k=heapq.heappop(hp)
   if not h[k]:continue
   cb=cash;h[k]=False;cash+=net[k];btc-=ba[k];real+=p[k];sfq+=sf[k];cy+=1;sold.add(k);eid+=1
   if abs(btc)<1e-12:btc=0.
   done.append({'grid_level':int(l[k]),'buy_time':bt[k],'sell_time':t,'buy_price':b[k],'sell_price':s[k],'cost':c[k],'base_amount':ba[k],'actual_earn':net[k],'grid_cashflow':p[k]});ev.append({'event_id':eid,'time':t,'side':'SELL','price':s[k],'cash_movement':net[k],'grid_cashflow':p[k],'cash_before':cb,'cash_after':cash});bt[k]=None
  budget=start;top=o if prev is None else max(prev,o)
  if lo<top:
   a,z=bisect.bisect_left(bl,lo),bisect.bisect_left(bl,top)
   for k in range(z-1,a-1,-1):
    if h[k] or k in sold:continue
    if budget+1e-12<c[k]:break
    cb=cash;h[k]=True;bt[k]=t;budget-=c[k];cash-=c[k];btc+=ba[k];bfb+=bf[k];bfq+=bf[k]*b[k];heapq.heappush(hp,(s[k],k));eid+=1;ev.append({'event_id':eid,'time':t,'side':'BUY','price':b[k],'cash_movement':-c[k],'grid_cashflow':0.,'cash_before':cb,'cash_after':cash})
  eq[i]=cash+btc*cl;cv[i]=cash;bv[i]=btc;prev=cl
 f,r,a,m,ca,dd=_stats(d,eq,initial_capital);curve=pd.DataFrame({'open_time':d.open_time,'close':d.close,'cash':cv,'btc':bv,'equity':eq,'drawdown':dd});log=pd.DataFrame(ev)
 if len(log):log['cumulative_cash_movement']=log.cash_movement.cumsum();log['cumulative_grid_cashflow']=log.grid_cashflow.cumsum()
 sm={'initial_capital':initial_capital,'final_equity':f,'net_return':r,'annualized_return':a,'max_drawdown':m,'calmar_ratio':ca,'completed_cycles':cy,'open_positions':int(h.sum()),'final_cash':cash,'final_btc':btc,'realized_profit':real,'unrealized_pnl':f-initial_capital-real,'buy_fee_btc':bfb,'buy_fee_usdt_equiv':bfq,'sell_fee_usdt':sfq,'total_fee_usdt_equiv':bfq+sfq}
 return {'summary':sm,'trade_log':log,'completed_trades':pd.DataFrame(done),'equity_curve':curve,'grid_state':g.assign(holding=h,buy_time=bt)}
def build_monthly_portfolio_pnl(equity_curve,initial_capital,start_date=None,end_date=None):
 e=equity_curve[['open_time','equity']].copy();e.open_time=pd.to_datetime(e.open_time,utc=True)
 if start_date:e=e[e.open_time>=pd.Timestamp(start_date,tz='UTC')]
 if end_date:e=e[e.open_time<pd.Timestamp(end_date,tz='UTC')]
 e['month']=e.open_time.dt.strftime('%Y-%m');m=e.groupby('month',as_index=False).equity.last().rename(columns={'equity':'ending_equity'});m['beginning_equity']=m.ending_equity.shift(1)
 if len(m):m.loc[0,'beginning_equity']=initial_capital
 m['portfolio_net_pnl']=m.ending_equity-m.beginning_equity;m['monthly_return']=m.portfolio_net_pnl/m.beginning_equity;m['cumulative_portfolio_pnl']=m.portfolio_net_pnl.cumsum();return m[['month','beginning_equity','ending_equity','portfolio_net_pnl','monthly_return','cumulative_portfolio_pnl']]
def round_to_gap(p,g):
 if g<=0:raise ValueError('gap must be greater than 0.')
 return float(np.floor(float(p)/g+.5)*g)
def build_dynamic_regime(reference_price,gap,number_of_grids,capital,regime_id=0):
 n=number_of_grids//2;f=float(reference_price-n*gap);ce=float(reference_price+(number_of_grids-n)*gap)
 if f<=0 or gap<=0 or capital<=0 or number_of_grids<2:raise ValueError('Invalid regime inputs.')
 return {'regime_id':regime_id,'reference_price':float(reference_price),'floor':f,'ceiling':ce,'buy_prices':np.arange(f,ce,gap,dtype=float),'capital_per_grid':capital/number_of_grids,'gap':gap}
def run_dynamic_grid_backtest(df_price,initial_capital,gap,number_of_grids,recenter_trigger_grids,buy_fee=.001,sell_fee=.001,min_cash_reserve=0.,max_open_positions=None,max_deployed_capital=None,max_entry_btc_exposure=None,max_drawdown_stop=None):
 d=df_price.sort_values('open_time').reset_index(drop=True);rid=0;rg=build_dynamic_regime(round_to_gap(d.iloc[0].open,gap),gap,number_of_grids,initial_capital);regs=[{'regime_id':0,'effective_time':d.iloc[0].open_time,'reference_price':rg['reference_price']}];rec=[];halts=[];cash=float(initial_capital);btc=dep=real=bfb=bfq=sfq=0.;cy=eid=pid=0;ps={};opx={};hp=[];ev=[];done=[];blocked={k:0 for k in ['risk_halt','cash_reserve','max_open_positions','max_deployed_capital','max_entry_btc_exposure']};n=len(d);eq=np.empty(n);cv=np.empty(n);bv=np.empty(n);dv=np.empty(n);ov=np.empty(n,int);mv=np.empty(n);refs=np.empty(n);rids=np.empty(n,int);rhs=np.empty(n,bool);prev=None;halt=False;peak=initial_capital;tol=1e-9
 for i,r in enumerate(d.itertuples(index=False)):
  t,o,hi,lo,cl=r.open_time,float(r.open),float(r.high),float(r.low),float(r.close);start=cash;sold=set()
  while hp and hp[0][0]<=hi+tol:
   _,q=heapq.heappop(hp);p=ps[q]
   if not p['open']:continue
   cb=cash;p['open']=False;cash+=p['net'];btc-=p['base'];dep-=p['cost'];real+=p['profit'];sfq+=p['sf'];cy+=1;sold.add(p['buy']);opx.pop(p['buy'],None);eid+=1;done.append({'position_id':q,'regime_id':p['rid'],'buy_time':p['time'],'sell_time':t,'buy_price':p['buy'],'sell_price':p['sell'],'cost':p['cost'],'base_amount':p['base'],'actual_earn':p['net'],'grid_cashflow':p['profit']});ev.append({'event_id':eid,'time':t,'side':'SELL','price':p['sell'],'cash_movement':p['net'],'grid_cashflow':p['profit'],'cash_before':cb,'cash_after':cash})
  budget=start;top=o if prev is None else max(prev,o);buys=rg['buy_prices'];bl=buys.tolist()
  if lo<top:
   a,z=bisect.bisect_left(bl,lo),bisect.bisect_left(bl,top)
   for k in range(z-1,a-1,-1):
    buy=float(buys[k]);cost=rg['capital_per_grid']
    if buy in opx or buy in sold:continue
    if halt:blocked['risk_halt']+=1;break
    if budget+tol<cost:break
    if budget-cost<min_cash_reserve-tol:blocked['cash_reserve']+=1;break
    if max_open_positions and len(opx)>=max_open_positions:blocked['max_open_positions']+=1;break
    if max_deployed_capital and dep+cost>max_deployed_capital+tol:blocked['max_deployed_capital']+=1;break
    sell=buy+gap;gross=cost/buy;bf=gross*buy_fee;base=gross-bf
    if max_entry_btc_exposure and (btc+base)*buy>max_entry_btc_exposure+tol:blocked['max_entry_btc_exposure']+=1;break
    gs=base*sell;sf=gs*sell_fee;net=gs-sf;profit=net-cost;cb=cash;budget-=cost;cash-=cost;btc+=base;dep+=cost;bfb+=bf;bfq+=bf*buy;pid+=1;ps[pid]={'rid':rg['regime_id'],'time':t,'buy':buy,'sell':sell,'cost':cost,'base':base,'sf':sf,'net':net,'profit':profit,'open':True};opx[buy]=pid;heapq.heappush(hp,(sell,pid));eid+=1;ev.append({'event_id':eid,'time':t,'side':'BUY','price':buy,'cash_movement':-cost,'grid_cashflow':0.,'cash_before':cb,'cash_after':cash})
  equity=cash+btc*cl;peak=max(peak,equity);dd=equity/peak-1
  if not halt and max_drawdown_stop and dd<=-max_drawdown_stop:halt=True;halts.append({'decision_time':t,'effective_time':d.iloc[i+1].open_time if i+1<n else pd.NaT,'drawdown':dd})
  eq[i]=equity;cv[i]=cash;bv[i]=btc;dv[i]=dep;ov[i]=len(opx);mv[i]=btc*cl;refs[i]=rg['reference_price'];rids[i]=rg['regime_id'];rhs[i]=halt
  if abs(cl-rg['reference_price'])+tol>=recenter_trigger_grids*gap:
   nr=round_to_gap(cl,gap)
   if nr!=rg['reference_price']:old=rg['reference_price'];rid+=1;rg=build_dynamic_regime(nr,gap,number_of_grids,initial_capital,rid);rec.append({'decision_time':t,'close':cl,'old_reference':old,'new_reference':nr});regs.append({'regime_id':rid,'effective_time':d.iloc[i+1].open_time if i+1<n else pd.NaT,'reference_price':nr})
  prev=cl
 f,r,a,m,ca,dd=_stats(d,eq,initial_capital);curve=pd.DataFrame({'open_time':d.open_time,'close':d.close,'cash':cv,'btc':bv,'btc_market_value':mv,'deployed_capital':dv,'open_positions':ov,'equity':eq,'reference_price':refs,'regime_id':rids,'risk_halt':rhs,'drawdown':dd});log=pd.DataFrame(ev)
 if len(log):log['cumulative_cash_movement']=log.cash_movement.cumsum();log['cumulative_grid_cashflow']=log.grid_cashflow.cumsum()
 opens=[p for p in ps.values() if p['open']];sm={'initial_capital':initial_capital,'final_equity':f,'net_return':r,'annualized_return':a,'max_drawdown':m,'calmar_ratio':ca,'completed_cycles':cy,'open_positions':len(opens),'final_cash':cash,'final_btc':btc,'final_deployed_capital':dep,'realized_profit':real,'unrealized_pnl':f-initial_capital-real,'buy_fee_btc':bfb,'buy_fee_usdt_equiv':bfq,'sell_fee_usdt':sfq,'total_fee_usdt_equiv':bfq+sfq,'recenter_count':len(rec),'risk_halt_triggered':halt,'risk_halt_count':len(halts),'min_cash_observed':float(cv.min()),'max_open_positions_observed':int(ov.max()),'max_deployed_capital_observed':float(dv.max()),'max_btc_market_value_observed':float(mv.max()),'blocked_buy_counts':blocked}
 return {'summary':sm,'trade_log':log,'completed_trades':pd.DataFrame(done),'equity_curve':curve,'open_positions':pd.DataFrame(opens),'recenter_log':pd.DataFrame(rec),'regime_history':pd.DataFrame(regs),'risk_halt_log':pd.DataFrame(halts)}

## 2. Run Backtests & Audit

In [ ]:
df_1m=load_market_data(SYMBOL,'1m',DATA_DIR);df_1m=df_1m[(df_1m.open_time>=pd.Timestamp(START_DATE,tz='UTC'))&(df_1m.open_time<pd.Timestamp(END_DATE,tz='UTC'))].reset_index(drop=True)
GRID_CAPITAL,GRID_CEILING,GRID_FLOOR,GRID_GAP=3000.,8987.,1987.,70.;assert np.isclose(build_excel_grid_table(GRID_CAPITAL,GRID_CEILING,GRID_FLOOR,GRID_GAP).iloc[0].profit,.1750644398340242,atol=1e-12)
BACKTEST_CAPITAL=3000.;BACKTEST_GAP=PRICE_ROUNDING=1000.;historical_low, historical_high=df_1m.low.min(),df_1m.high.max();BACKTEST_FLOOR=np.floor(historical_low/1000)*1000;BACKTEST_CEILING=np.ceil(historical_high/1000)*1000;NUMBER_OF_GRIDS=int((BACKTEST_CEILING-BACKTEST_FLOOR)/BACKTEST_GAP);CAPITAL_PER_LEVEL=BACKTEST_CAPITAL/NUMBER_OF_GRIDS
v0=run_grid_backtest(df_1m,build_excel_grid_table(BACKTEST_CAPITAL,BACKTEST_CEILING,BACKTEST_FLOOR,BACKTEST_GAP),BACKTEST_CAPITAL);v0_summary=v0['summary']
V1_GRID_GAP=1000.;V1_NUMBER_OF_GRIDS=30;V1_RECENTER_TRIGGER_GRIDS=5;V1_CAPITAL_PER_GRID=100.;V1_MIN_CASH_RESERVE=750.;V1_MAX_OPEN_POSITIONS=18;V1_MAX_DEPLOYED_CAPITAL=1800.;V1_MAX_ENTRY_BTC_EXPOSURE=1800.;V1_MAX_DRAWDOWN_STOP=.15;LIVE_EXECUTION_ENABLED=False;EXECUTION_MODE='BACKTEST / SHADOW READINESS'
v1=run_dynamic_grid_backtest(df_1m,BACKTEST_CAPITAL,V1_GRID_GAP,V1_NUMBER_OF_GRIDS,V1_RECENTER_TRIGGER_GRIDS,.001,.001,V1_MIN_CASH_RESERVE,V1_MAX_OPEN_POSITIONS,V1_MAX_DEPLOYED_CAPITAL,V1_MAX_ENTRY_BTC_EXPOSURE,V1_MAX_DRAWDOWN_STOP);v1_summary=v1['summary']
spec=[('Final Equity','final_equity',1),('Net Return %','net_return',100),('Max Drawdown %','max_drawdown',100),('Calmar','calmar_ratio',1),('Cycles','completed_cycles',1),('Open Positions','open_positions',1)]
df_comparison=pd.DataFrame([{'Metric':a,'V0':v0_summary[k]*s,'V1':v1_summary[k]*s,'Difference':(v1_summary[k]-v0_summary[k])*s} for a,k,s in spec]);display(df_comparison)
def audit(r,name):
 s,l,e=r['summary'],r['trade_log'],r['equity_curve'];cm=l.cash_movement.sum() if len(l) else 0.;gf=l.grid_cashflow.sum() if len(l) else 0.;z=[abs(BACKTEST_CAPITAL+cm-s['final_cash']),abs(gf-s['realized_profit']),float(np.max(np.abs(e.cash+e.btc*e.close-e.equity)))]
 return [{'Strategy':name,'Test':t,'Actual':x,'Status':'PASS' if x<=1e-8 else 'FAIL'} for t,x in zip(['Cash reconciliation','Grid cashflow reconciliation','Equity identity'],z)]
aud=audit(v0,'V0')+audit(v1,'V1');aud+=[{'Strategy':'V1','Test':'Cash reserve','Actual':v1_summary['min_cash_observed'],'Status':'PASS' if v1_summary['min_cash_observed']>=750-1e-8 else 'FAIL'},{'Strategy':'V1','Test':'Max positions','Actual':v1_summary['max_open_positions_observed'],'Status':'PASS' if v1_summary['max_open_positions_observed']<=18 else 'FAIL'},{'Strategy':'V1','Test':'Max deployed','Actual':v1_summary['max_deployed_capital_observed'],'Status':'PASS' if v1_summary['max_deployed_capital_observed']<=1800+1e-8 else 'FAIL'}]
df_system_test_log=pd.DataFrame(aud);display(df_system_test_log);SYSTEM_AUDIT_STATUS='PASS' if df_system_test_log.Status.eq('PASS').all() else 'FAIL'
if SYSTEM_AUDIT_STATUS!='PASS':raise AssertionError('SYSTEM AUDIT FAILED')
print(f"V0 {v0_summary['net_return']:.2%} / DD {v0_summary['max_drawdown']:.2%} | V1 {v1_summary['net_return']:.2%} / DD {v1_summary['max_drawdown']:.2%} | Audit {SYSTEM_AUDIT_STATUS}")

## 3. Export Log

In [ ]:
def sf(v):
 if isinstance(v,(np.integer,np.floating)):return v.item()
 if isinstance(v,pd.Timestamp):return v.isoformat()
 return v
def rc(d):return [{k:sf(v) for k,v in x.items()} for x in d.to_dict('records')]
log_payload={'log_schema_version':4,'run_info':{'generated_at_utc':pd.Timestamp.now(tz='UTC').isoformat(),'symbol':SYMBOL,'start_date':START_DATE,'end_date':END_DATE,'data_rows':len(df_1m)},'v0_summary':{k:sf(v) for k,v in v0_summary.items()},'v1_summary':{k:(sf(v) if not isinstance(v,dict) else v) for k,v in v1_summary.items()},'system_audit':{'status':SYSTEM_AUDIT_STATUS,'checks':rc(df_system_test_log)},'comparison':rc(df_comparison)}
with open('/content/latest_backtest_log.json','w') as f:json.dump(log_payload,f,indent=2)
try:
 from google.colab import userdata
 token=userdata.get('GITHUB_TOKEN')
except:token=None
if token:
 import requests
 u='https://api.github.com/repos/natdanaiii/Trading/contents/logs/latest_backtest_log.json';h={'Authorization':f'Bearer {token}'};o=requests.get(u,headers=h,timeout=30);b={'message':'Update latest backtest log','content':base64.b64encode(json.dumps(log_payload,indent=2).encode()).decode(),'branch':'main'}
 if o.status_code==200:b['sha']=o.json()['sha']
 q=requests.put(u,headers=h,json=b,timeout=30);q.raise_for_status();print('GitHub log upload: SUCCESS')